<a href="https://colab.research.google.com/github/vishal9198/genAi-Labs/blob/main/multiRepresentation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ==============================================================================
# Cell 1: Install Dependencies
# ==============================================================================
!pip install -qU langchain langchain-community langchain-core langchain-openai langchain-chroma tiktoken beautifulsoup4

# ==============================================================================
# Cell 2: API Key Setup (Using Colab Secrets)
# ==============================================================================
import os
from google.colab import userdata

try:
    os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
except Exception:
    import getpass
    os.environ["OPENAI_API_KEY"] = getpass.getpass("Enter OpenAI API Key: ")

# ==============================================================================
# Cell 3: Document Loading, Summarization, and Multi-Vector Indexing
# ==============================================================================
import uuid
import bs4
from langchain_community.document_loaders import WebBaseLoader
from langchain_core.documents import Document
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_community.vectorstores import Chroma
from langchain.storage import InMemoryByteStore
from langchain.retrievers.multi_vector import MultiVectorRetriever

# 1. Load two different web articles as full raw documents
loader1 = WebBaseLoader(
    web_paths=("https://lilianweng.github.io/posts/2023-06-23-agent/",),
    bs_kwargs=dict(parse_only=bs4.SoupStrainer(class_=("post-content", "post-title", "post-header")))
)
docs = loader1.load()

loader2 = WebBaseLoader(
    web_paths=("https://lilianweng.github.io/posts/2024-02-05-human-data-quality/",),
    bs_kwargs=dict(parse_only=bs4.SoupStrainer(class_=("post-content", "post-title", "post-header")))
)
docs.extend(loader2.load())
print(f"Loaded {len(docs)} parent documents.")

# 2. Build a summarization chain using an LLM
summarize_prompt = ChatPromptTemplate.from_template("Summarize the following document concisely:\n\n{doc}")
summarize_chain = (
    {"doc": lambda x: x.page_content}
    | summarize_prompt
    | ChatOpenAI(model="gpt-3.5-turbo", temperature=0)
    | StrOutputParser()
)

# Generate dense summaries concurrently
summaries = summarize_chain.batch(docs, {"max_concurrency": 5})

# 3. Configure MultiVectorRetriever
vectorstore = Chroma(
    collection_name="summaries",
    embedding_function=OpenAIEmbeddings()
)
store = InMemoryByteStore()
id_key = "doc_id"

retriever = MultiVectorRetriever(
    vectorstore=vectorstore,
    byte_store=store,
    id_key=id_key,
)

# 4. Generate unique IDs and link summaries to parent documents
doc_ids = [str(uuid.uuid4()) for _ in docs]

# Wrap summaries as Documents with doc_id metadata
summary_docs = [
    Document(page_content=s, metadata={id_key: doc_ids[i]})
    for i, s in enumerate(summaries)
]

# Add summaries to the vector database for search
retriever.vectorstore.add_documents(summary_docs)

# Add full original documents to the key-value document store
retriever.docstore.mset(list(zip(doc_ids, docs)))
print("Indexing complete: Summaries in VectorStore, Full Docs in DocStore.")

# ==============================================================================
# Cell 4: Retrieval and Generation Test
# ==============================================================================
query = "Memory in agents"

# Inspect the vector store match (returns the summary)
matched_summary = vectorstore.similarity_search(query, k=1)
print("\n--- MATCHED SUMMARY IN VECTORSTORE ---")
print(f"Doc ID: {matched_summary[0].metadata['doc_id']}")
print(f"Summary Content:\n{matched_summary[0].page_content}")

# Inspect the MultiVectorRetriever output (fetches the full parent document)
retrieved_parent_docs = retriever.invoke(query)
print("\n--- RETRIEVED FULL PARENT DOCUMENT (First 500 chars) ---")
print(retrieved_parent_docs[0].page_content[:500].strip())